In [1]:
import zipfile
import os
import torch
import torch.nn as nn
import torch.optim as optim
import pandas as pd
import numpy as np
from sklearn.preprocessing import StandardScaler
from sklearn.utils.class_weight import compute_class_weight
from sklearn.metrics import classification_report, balanced_accuracy_score
from sklearn.model_selection import train_test_split
from torch.utils.data import DataLoader, TensorDataset

In [2]:
label_sale_price = lambda amount: 0 if amount <= 100000 else 2 if amount > 350000 else 1

train_data, test_data = (
    pd.read_csv('train_data.csv'),
    pd.read_csv('test_data.csv'),
)

In [3]:
# create class and delete SalePrice

train_data['Target'] = train_data['SalePrice'].apply(label_sale_price)

train_data = train_data.drop('SalePrice', axis=1)

train_data

,YearBuilt,Size(sqf),Floor,HallwayType,HeatingType,AptManageType,N_Parkinglot(Ground),N_Parkinglot(Basement),TimeToBusStop,TimeToSubway,N_manager,N_elevators,SubwayStation,N_FacilitiesInApt,N_FacilitiesNearBy(Total),N_SchoolNearBy(Total),Target
0,2006,814,3,terraced,individual_heating,management_in_trust,111.0,184.0,5min~10min,10min~15min,3.0,0.0,Kyungbuk_uni_hospital,5,6.0,9.0,1
1,1985,587,8,corridor,individual_heating,self_management,80.0,76.0,0~5min,5min~10min,2.0,2.0,Daegu,3,12.0,4.0,0
2,1985,587,6,corridor,individual_heating,self_management,80.0,76.0,0~5min,5min~10min,2.0,2.0,Daegu,3,12.0,4.0,0
3,2006,2056,8,terraced,individual_heating,management_in_trust,249.0,536.0,0~5min,0-5min,5.0,11.0,Sin-nam,5,3.0,7.0,2
4,1992,644,2,mixed,individual_heating,self_management,142.0,79.0,5min~10min,15min~20min,4.0,8.0,Myung-duk,3,9.0,14.0,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
4119,2007,1928,24,terraced,individual_heating,management_in_trust,0.0,1270.0,0~5min,0-5min,14.0,16.0,Kyungbuk_uni_hospital,10,9.0,10.0,2
4120,2015,644,22,terraced,individual_heating,management_in_trust,102.0,400.0,0~5min,5min~10min,5.0,10.0,Daegu,7,7.0,11.0,1
4121,2007,868,20,terraced,individual_heating,management_in_trust,0.0,1270.0,0~5min,0-5min,14.0,16.0,Kyungbuk_uni_hospital,10,9.0,10.0,2
4122,1978,1327,1,corridor,individual_heating,self_management,87.0,0.0,0~5min,0-5min,1.0,4.0,Kyungbuk_uni_hospital,3,7.0,11.0,1


In [4]:
# target variable
y_train = train_data['Target']
X_train = train_data.drop('Target', axis=1)
X_test = test_data.copy()

# concat sets to link categories in the same manner
train_len = len(train_data)
all_data = pd.concat([X_train, X_test], axis=0).reset_index(drop=True)

all_data

,YearBuilt,Size(sqf),Floor,HallwayType,HeatingType,AptManageType,N_Parkinglot(Ground),N_Parkinglot(Basement),TimeToBusStop,TimeToSubway,N_manager,N_elevators,SubwayStation,N_FacilitiesInApt,N_FacilitiesNearBy(Total),N_SchoolNearBy(Total)
0,2006,814,3,terraced,individual_heating,management_in_trust,111.0,184.0,5min~10min,10min~15min,3.0,0.0,Kyungbuk_uni_hospital,5,6.0,9.0
1,1985,587,8,corridor,individual_heating,self_management,80.0,76.0,0~5min,5min~10min,2.0,2.0,Daegu,3,12.0,4.0
2,1985,587,6,corridor,individual_heating,self_management,80.0,76.0,0~5min,5min~10min,2.0,2.0,Daegu,3,12.0,4.0
3,2006,2056,8,terraced,individual_heating,management_in_trust,249.0,536.0,0~5min,0-5min,5.0,11.0,Sin-nam,5,3.0,7.0
4,1992,644,2,mixed,individual_heating,self_management,142.0,79.0,5min~10min,15min~20min,4.0,8.0,Myung-duk,3,9.0,14.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
5886,2006,2056,2,terraced,individual_heating,management_in_trust,249.0,536.0,0~5min,0-5min,5.0,11.0,Sin-nam,5,3.0,7.0
5887,2007,1394,7,terraced,individual_heating,management_in_trust,554.0,524.0,0~5min,0-5min,5.0,10.0,Banwoldang,4,9.0,8.0
5888,1993,644,20,mixed,individual_heating,management_in_trust,523.0,536.0,0~5min,15min~20min,8.0,20.0,Myung-duk,4,14.0,17.0
5889,2008,914,11,terraced,individual_heating,management_in_trust,197.0,475.0,5min~10min,0-5min,6.0,14.0,Sin-nam,8,7.0,9.0


In [5]:
# filling numeric columns with median and text with None

numeric_cols    = all_data.select_dtypes(include=['int64', 'float64']).columns
categorial_cols = all_data.select_dtypes(include=['object']).columns

all_data[numeric_cols] = all_data[numeric_cols].fillna(
    all_data[numeric_cols].median()
)

all_data[categorial_cols] = all_data[categorial_cols].fillna('None')


In [6]:
# hot one encoding, cleaning up and data scaling for MLP

all_data = pd.get_dummies(all_data)

scaler = StandardScaler()

X_train_clean = all_data.iloc[:train_len, :]
X_test_clean = all_data.iloc[train_len:, :]

X_train_scaled = scaler.fit_transform(X_train_clean)
X_test_scaled = scaler.transform(X_test_clean)

In [7]:
class HousePriceNet(nn.Module):
    def __init__(self, input_size):
        super(HousePriceNet, self).__init__()

        self.net = nn.Sequential(
            nn.Linear(input_size, 128),
            nn.BatchNorm1d(128),
            nn.LeakyReLU(0.01),
            nn.Dropout(0.3),

            nn.Linear(128, 64),
            nn.BatchNorm1d(64),
            nn.LeakyReLU(0.01),
            nn.Dropout(0.2),

            nn.Linear(64, 3)    # 3 because (0, 1, 2) = (cheap, medium, expensive)
        )

    def forward(self, x):
        return self.net(x)

In [9]:
X_train_split, X_val_split, y_train_split, y_val_split = train_test_split(
    X_train_scaled,
    y_train.values,
    test_size=0.2,
    random_state=42,
    stratify=y_train.values,
)

X_train_tensor, y_train_tensor = (
    torch.FloatTensor(X_train_split),
    torch.LongTensor(y_train_split)
)

X_val_tensor, y_val_tensor = (
    torch.FloatTensor(X_val_split),
    torch.LongTensor(y_val_split),
)

dataset = TensorDataset(X_train_tensor, y_train_tensor)
dataloader = DataLoader(dataset, batch_size=32, shuffle=True)

class_weights = compute_class_weight('balanced', classes=np.unique(y_train), y=y_train)
weights_tensor = torch.FloatTensor(class_weights)

weights_tensor

tensor([2.4460, 0.4594, 2.4117])

In [10]:
num_cols = X_train_scaled.shape[1]
model = HousePriceNet(input_size=num_cols)

# loss function
criterion = nn.CrossEntropyLoss(weight=weights_tensor)

# adam optimizer
learning_rate = 0.001
optimizer = optim.Adam(model.parameters(), lr=learning_rate)

# learn loop
epochs = 50
model.train()

for epoch in range(epochs):
    model.train()
    epoch_loss = 0.0
    for batch_X, batch_y in dataloader:
        optimizer.zero_grad()
        outputs = model(batch_X)
        loss = criterion(outputs, batch_y)
        loss.backward()
        optimizer.step()
        epoch_loss += loss.item()

    # validation
    model.eval()
    with torch.no_grad():
        val_outputs = model(X_val_tensor)
        val_loss = criterion(val_outputs, y_val_tensor)
        _, val_predictions = torch.max(val_outputs, 1)

        # balanced accuracy for validation
        val_bal_acc = balanced_accuracy_score(y_val_tensor.numpy(), val_predictions.numpy())

    if (epoch + 1) % 10 == 0:
        print(f"Epoch [{epoch + 1}/{epochs}] | "
              f"Train Loss: {epoch_loss/len(dataloader):.4f} | "
              f"Val Loss: {val_loss.item():.4f} | "
              f"Val Balanced Acc: {val_bal_acc*100:.2f}%")

print("Training complete")

Epoch [10/50] | Train Loss: 0.3403 | Val Loss: 0.3106 | Val Balanced Acc: 88.76%
Epoch [20/50] | Train Loss: 0.3046 | Val Loss: 0.2946 | Val Balanced Acc: 89.26%
Epoch [30/50] | Train Loss: 0.3261 | Val Loss: 0.2898 | Val Balanced Acc: 88.45%
Epoch [40/50] | Train Loss: 0.3224 | Val Loss: 0.2998 | Val Balanced Acc: 88.60%
Epoch [50/50] | Train Loss: 0.3034 | Val Loss: 0.2950 | Val Balanced Acc: 89.41%
Training complete


In [11]:
model.eval()

X_test_tensor = torch.FloatTensor(X_test_scaled)

with torch.no_grad():
    test_outputs = model(X_test_tensor)
    _, predictions = torch.max(test_outputs, 1)

final_preds = predictions.numpy()

# save predictions to pred.csv (no header, one integer column)
pd.DataFrame(final_preds).to_csv('pred.csv', index=False, header=False)

print(f"Saved {len(final_preds)} predictions to pred.csv")
print(f"Class distribution: {pd.Series(final_preds).value_counts().sort_index().to_dict()}")


Saved 1767 predictions to pred.csv
Class distribution: {0: 404, 1: 987, 2: 376}


In [12]:
model.eval()

with torch.no_grad():
    val_outputs = model(X_val_tensor)
    _, val_predictions = torch.max(val_outputs, 1)

val_preds_np = val_predictions.numpy()
y_val_np = y_val_tensor.numpy()

print("\n--- Report on validation set ---")
print("Balanced Accuracy:")
print(f"{balanced_accuracy_score(y_val_np, val_preds_np) * 100:.2f}%\n")

print("Detailed classification report:")
print(classification_report(y_val_np, val_preds_np))


--- Report on validation set ---
Balanced Accuracy:
89.41%

Detailed classification report:
              precision    recall  f1-score   support

           0       0.58      0.99      0.73       112
           1       0.99      0.73      0.84       599
           2       0.57      0.96      0.71       114

    accuracy                           0.80       825
   macro avg       0.71      0.89      0.76       825
weighted avg       0.88      0.80      0.81       825



In [13]:
# pack into zip file

DAY = "sroda"
AUTHOR_1 = "BagińskiJakub"
AUTHOR_2 = "GórskiKacper"
CODE_FILE = "cw2.ipynb"

autorzy = sorted([AUTHOR_1, AUTHOR_2])
nazwa_zip = f"{DAY}_{autorzy[0]}_{autorzy[1]}.zip"

with zipfile.ZipFile(nazwa_zip, 'w') as zipf:
    if os.path.exists('pred.csv'):
        zipf.write('pred.csv')
    else:
        print("ERROR: pred.csv not found!")

    if os.path.exists(CODE_FILE):
        zipf.write(CODE_FILE)
        print(f"Package ready: {nazwa_zip}")
    else:
        print(f"ERROR: {CODE_FILE} not found!")



ERROR: cw2.ipynb not found!
Package ready: sroda_BagińskiJakub_GórskiKacper.zip
